In [24]:
import os
import nd2
import napari
from tifffile import imwrite, imread

from pathlib import Path

from glob import glob

import dask.array as da

from tqdm.notebook import trange, tqdm

import numpy as np
from numpy.fft import rfftn, irfftn, fftshift

from scipy.ndimage import affine_transform, shift, gaussian_filter, convolve

In [12]:
def fft_translation(img0, img1):
    fft_img0 = rfftn(img0)
    fft_img1 = rfftn(img1)
    fft_img1_cc = np.conjugate(fft_img1)

    mult = fft_img0 * fft_img1_cc
    mult /= np.abs(mult)
    inverse = irfftn(mult, s=img0.shape, axes=range(img0.ndim))

    inverse_shifted = fftshift(inverse)
    # Offset relative to the center of the array
    peak = np.array(np.unravel_index(np.argmax(inverse_shifted), shape=img0.shape))
    offset = peak - np.array(img0.shape) // 2

    return offset

def fft_translation_nonzero(img0, img1):
    fft_img0 = rfftn(img0)
    fft_img1 = rfftn(img1)
    fft_img1_cc = np.conjugate(fft_img1)

    mult = fft_img0 * fft_img1_cc
    mult /= np.abs(mult)
    inverse = irfftn(mult, s=img0.shape, axes=range(img0.ndim))

    inverse_shifted = fftshift(inverse)
    # Offset relative to the center of the array
    vals = np.argsort(inverse_shifted.ravel())[::-1]
    print(vals.shape)
    i = 0
    offset = np.zeros(3)
    while offset[1] == offset[2] == 0 or offset[0] == 0:
        peak = np.array(np.unravel_index(vals[i], shape=img0.shape))
        print(peak, i)
        offset = peak - np.array(img0.shape) // 2
        i += 1

    return offset

In [31]:
def mask_dead_pixels(img, mask):
    weight = np.zeros((3, 3))
    weight[1, 1] = 0
    mean = convolve(img, weight) / weight.sum()
    masked = img.copy()
    masked[mask] = mean[mask]
    return masked

def dead_pixel_values(img, mask):
    weight = np.ones((3, 3))
    weight[1, 1] = 0
    mean = (convolve(img, weight) / weight.sum()).astype(img.dtype)
    return mean[mask]

def mask_dead_pixels_3d(img, mask):
    new = np.copy(img)
    for i in range(len(img)):
        new[i, mask] = dead_pixel_values(img[i], mask)
    return new

In [3]:
viewer = napari.Viewer()

In [4]:
path = "/mnt/z/Dasha/2. FAST-AB biosensors/Microscope/Inverted bladder model/TrimFABS sensor 30.06.2026/To analyze"

In [33]:
dead_pixel_mask = imread("/mnt/f/pixel-conv.tif")[0] > 25
mean_values = imread("/mnt/f/pixel-mean.tif")[0]

In [5]:
if False:
    file = "NCCRBM128_sample4.nd2"
    
    files = [
        "NCCRBM128_sample4_before.nd2",
        "NCCRBM128_sample4_TMP.nd2",
    ]
    
    imgs = [
        nd2.ND2File(os.path.join(path, file))
        for file in files
    ]
    
    img_da = da.stack([img.to_dask() for img in imgs], axis=0)
else:
    file = "NCCRBM128_sample5.nd2"
    file = "NCCRBM128_sample6.nd2"
    
    nd2_file = nd2.ND2File(os.path.join(path, file))
    img_da = nd2_file.to_dask()

In [5]:
nd2_file.sizes

mappingproxy({'T': 2, 'P': 4, 'Z': 35, 'C': 3, 'Y': 2304, 'X': 2304})

In [80]:
def create_slices(offset, img_shape):
    slices0 = [slice(None)]
    slices1 = [slice(None)]
    shapes = [img_shape[0], img_shape[2], img_shape[3]]
    for i, o in enumerate(offset):
        if o >= 0:
            slices0.append(slice(o, shapes[i]))
            slices1.append(slice(0, shapes[i] - o))
        else:
            slices1.append(slice(-o, shapes[i]))
            slices0.append(slice(0, shapes[i] + o))
    
    return tuple(slices0), tuple(slices1)

In [81]:
for i in trange(4):
    img0 = img_da[0, i].compute()
    img1 = img_da[1, i].compute()

    offset = fft_translation(img0[:, 0], img1[:, 0])

    slices0, slices1 = create_slices(offset, img1.shape)

    x0 = img0.transpose(1, 0, 2, 3)[slices0]
    x1 = img1.transpose(1, 0, 2, 3)[slices1]

    new_img = np.stack([x0, x1], axis=0)

    imwrite(
        os.path.join(path, "tif-files", file.replace(".nd2", f"_pos{i}.tif")),
        new_img,
    )

  0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
def create_pads(offset, img_shape):
    pads0 = [(0, 0)]
    pads1 = [(0, 0)]
    shapes = [img_shape[0], img_shape[2], img_shape[3]]
    for i, o in enumerate(offset):
        if o >= 0:
            pads0.append((0, o))
            pads1.append((o, 0))
        else:
            pads1.append((0, -o))
            pads0.append((-o, 0))
    
    return pads0, pads1

def perc_norm(x, q0=5, q1=95):
    mi, ma = np.percentile(x, q=(q0, q1))
    return (x - mi) / (ma - mi)

In [39]:
#for i in trange(4):
for i in [3]:
    img0 = img_da[0, i].compute()
    img1 = img_da[1, i].compute()

    if "sample5" in file and i == 3:
        print("sample5_pos3")
        offset = fft_translation(img0[11:26, 0], img1[:15, 0])
        offset[0] += 11
    elif "sample6" in file and i == 2:
        print("sample6_pos2")
        sl = (slice(None), 0, slice(1800, img0.shape[-2]), slice(0, 600))
        offset = fft_translation_nonzero(img0[sl], img1[sl])
    elif "sample6" in file and i == 3:
        print("sample6_pos3")
        #sl = (slice(None), 0, slice(1800, img0.shape[-2]), slice(0, 600))
        #tmp0 = mask_dead_pixels_3d(img0[:, 0]-mean_values, dead_pixel_mask)
        #tmp1 = mask_dead_pixels_3d(img1[:, 0]-mean_values, dead_pixel_mask)
        #print(img1.shape)
        #print(tmp1.shape)
        #offset = fft_translation(tmp0, tmp1)
        offset = np.array((12, 1157, 1152)) - np.array(img0[:, 0].shape) // 2
        #offset = np.array((7, 1159, 1152)) - np.array(img0[:, 0].shape) // 2
    else:
        offset = fft_translation(img0[:, 0], img1[:, 0])

    print(offset)
    pads0, pads1 = create_pads(offset, img1.shape)
    print(pads0, pads1)

    x0 = np.pad(img0.transpose(1, 0, 2, 3), pads0, mode="symmetric")
    x1 = np.pad(img1.transpose(1, 0, 2, 3), pads1, mode="symmetric")

    zeros = np.ones((img0.shape[0], img0.shape[2], img0.shape[3]), dtype=bool)
    mask0 = np.pad(zeros, pads0[1:], mode="constant", constant_values=0)
    mask1 = np.pad(zeros, pads1[1:], mode="constant", constant_values=0)

    mask = np.logical_and(mask0, mask1)

    new_img = np.stack([x0, x1], axis=0)

    imwrite(
        os.path.join(path, "padded-tif-files", file.replace(".nd2", f"_pos{i}.tif")),
        new_img,
    )

    imwrite(
        os.path.join(path, "padded-tif-files", file.replace(".nd2", f"_pos{i}_mask.tif")),
        mask,
    )

sample6_pos3
[-5  5  0]
[(0, 0), (np.int64(5), 0), (0, np.int64(5)), (0, np.int64(0))] [(0, 0), (0, np.int64(5)), (np.int64(5), 0), (np.int64(0), 0)]


In [ ]:
img0[:, 0].max(), img0[:, 0].min(), img1[:, 0].max(), img1[:, 0].min()

In [ ]:
pads0, pads1, offset

In [91]:
tif_files = glob(os.path.join(path, "padded-tif-files", "*pos*.tif"))
for file in tqdm(tif_files):
    img = imread(file)
    name = Path(file).stem
    for i in range(2):
        imwrite(
            os.path.join(path, "padded-tif-files", "sciCORE", f"{name}_t{i}_mScarlett.tiff"),
            img[i, 1],
        )

  0%|          | 0/12 [00:00<?, ?it/s]